In [1]:
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

In [2]:
station_spacings = pd.read_csv("../data/processed/stations_delay_ridership_joined.csv")
sj_station_adt = pd.read_csv("../data/processed/stations_delay_ridership_joined_with_sj_adt.csv")

# Regression Models

In [3]:
#model 1: boardings and stop spacings for all selected routes
ind_vars = [
    'boardings', 
    'spacing_ft'   
]

X = station_spacings[ind_vars].copy()
Y = station_spacings['mean_delay']

#scale independent variables
X[ind_vars] = StandardScaler().fit_transform(X[ind_vars])

# Add a constant to the independent variables
X = sm.add_constant(X)
# Fit the linear regression model
model_boardings_spacings = sm.OLS(Y, X).fit()

print(model_boardings_spacings.summary())

                            OLS Regression Results                            
Dep. Variable:             mean_delay   R-squared:                       0.023
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     8.162
Date:                Tue, 12 May 2026   Prob (F-statistic):           0.000313
Time:                        09:38:58   Log-Likelihood:                -4594.3
No. Observations:                 710   AIC:                             9195.
Df Residuals:                     707   BIC:                             9208.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -28.1069      5.879     -4.781      0.0

In [4]:
#model 2: boardings, stop spacings, ADT for stations where data is available (City of San Jose only)
ind_vars = [
    'boardings', 
    'spacing_ft',
    'adt'
]

X = sj_station_adt[ind_vars].copy()
Y = sj_station_adt['mean_delay']

#scale independent variables
X[ind_vars] = StandardScaler().fit_transform(X[ind_vars])

# Add a constant to the independent variables
X = sm.add_constant(X)
# Fit the linear regression model
model_boardings_spacings_adt = sm.OLS(Y, X).fit()

print(model_boardings_spacings_adt.summary())

                            OLS Regression Results                            
Dep. Variable:             mean_delay   R-squared:                       0.057
Model:                            OLS   Adj. R-squared:                  0.046
Method:                 Least Squares   F-statistic:                     5.053
Date:                Tue, 12 May 2026   Prob (F-statistic):            0.00205
Time:                        09:38:58   Log-Likelihood:                -1646.4
No. Observations:                 255   AIC:                             3301.
Df Residuals:                     251   BIC:                             3315.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.7865      9.725      0.081      0.9

Boardings appears to be the most significant factor in both models, with an inverse correlation, which is interesting. Less boardings, more delay? Spacings make intuitive sense, even if its considerably less significant. Smaller stop spacings, more stops, so more delay. Finally, ADT also makes intuitive sense, as more traffic results in more delay.

# Robustness check: Spearman & Pearson correlations

Per instructor feedback, we report **rank-based (Spearman) and linear (Pearson) correlations** alongside the OLS regression. Spearman is robust to skewed boardings distributions and the long tail of stop spacings on Route 25.

We exclude `spacing_ft == 0` rows (consecutive stops mapped to the same OSM node, ~25 cases) and `boardings == 0` rows; otherwise the full station sample is used.

In [5]:
from scipy import stats

def corr_table(df, y_col, x_cols, label):
    rows = []
    for x in x_cols:
        sub = df[[x, y_col]].dropna()
        sub = sub[sub[x] > 0]
        pr, pp = stats.pearsonr(sub[x], sub[y_col])
        sr, sp = stats.spearmanr(sub[x], sub[y_col])
        rows.append({
            'pair': f'{x} vs. {y_col}',
            'n': len(sub),
            'pearson_r': round(pr, 3),
            'pearson_p': f'{pp:.2e}',
            'spearman_rho': round(sr, 3),
            'spearman_p': f'{sp:.2e}',
        })
    out = pd.DataFrame(rows)
    print(f'--- {label} ---')
    print(out.to_string(index=False))
    print()
    return out

corr_all = corr_table(
    station_spacings,
    y_col='mean_delay',
    x_cols=['boardings', 'spacing_ft'],
    label='Model 1 sample (all selected routes)'
)

corr_sj = corr_table(
    sj_station_adt,
    y_col='mean_delay',
    x_cols=['boardings', 'spacing_ft', 'adt'],
    label='Model 2 sample (San Jose stops with ADT)'
)

--- Model 1 sample (all selected routes) ---
                     pair   n  pearson_r pearson_p  spearman_rho spearman_p
 boardings vs. mean_delay 707     -0.142  1.47e-04        -0.148   8.20e-05
spacing_ft vs. mean_delay 685     -0.021  5.75e-01        -0.216   1.17e-08

--- Model 2 sample (San Jose stops with ADT) ---
                     pair   n  pearson_r pearson_p  spearman_rho spearman_p
 boardings vs. mean_delay 255     -0.180  3.93e-03        -0.115   6.69e-02
spacing_ft vs. mean_delay 246     -0.076  2.36e-01        -0.235   2.01e-04
       adt vs. mean_delay 255      0.120  5.58e-02         0.173   5.62e-03



**Reading the table:**

- `spacing_ft`: Pearson r is near zero, but Spearman ρ ≈ -0.22 (p ≪ 0.001). The Spearman result indicates a **robust monotonic** association — closer stops are paired with longer delays — that the linear Pearson statistic misses because of long-tail outliers on Route 25 segments crossing freeway gaps.
- `boardings`: weak but consistent negative association in both Pearson and Spearman. The unexpected sign likely reflects that high-ridership stops sit on Rapid 522/523 corridors, which run looser timetables and therefore tend to arrive ahead of schedule.
- `adt`: positive association in the San Jose subsample (more traffic → more delay), consistent with intuition.

We treat all three as **descriptive associations**, not causal effects, given the day-specific RT-data sample and the imperfect schedule-to-observation matching documented in Section 3.